# Phase 4: Few-Shot LLM API Baseline

**CSCI E-222 · Spring 2026**

Evaluates GPT-4o-mini and Claude Haiku as zero-training-cost baselines for multi-label product tagging.
Both models serve as the OOD fallback in the live pipeline (Phase 5).

This notebook:
1. Inspects the 5-shot prompt
2. Runs both providers on the test set (responses cached to disk)
3. Logs all metrics — same set as Phases 2 & 3 for direct comparison
4. Documents failure modes: hallucinated tags, parse failures, latency variance
5. Estimates per-call cost at scale

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import logging
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from models.llm_api import LLMApiTagger, FEW_SHOT_EXAMPLES
from eval.metrics import compute_metrics, per_label_f1, compare_models

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
sns.set_theme(style='whitegrid', palette='muted')

with open('../data/taxonomy.json') as f:
    taxonomy = json.load(f)
LABEL_NAMES = [l['tag'] for l in taxonomy['labels']]
NUM_LABELS  = len(LABEL_NAMES)

print(f'Labels: {NUM_LABELS}')

## 1. Inspect the 5-Shot Prompt

Review the prompt before running any API calls.

In [ ]:
# Instantiate without making any API calls
tagger = LLMApiTagger(
    provider='anthropic',
    model='claude-haiku-4-5-20251001',
    taxonomy_path='../data/taxonomy.json',
    cache_dir='../data/raw/llm_cache',
)

sample_product = {'title': 'Example Product', 'description': 'A sample product for prompt inspection.'}
prompt = tagger._build_prompt(sample_product, FEW_SHOT_EXAMPLES)
print(prompt)

## 2. Smoke Test (5 Products)

Verify the pipeline end-to-end before running the full test set.

In [ ]:
import pandas as pd
test_df = pd.read_parquet('../data/processed/test.parquet')
smoke   = test_df.sample(5, random_state=0).to_dict('records')

print(f'Running smoke test on 5 products with Claude Haiku...\n')
for product in smoke:
    result = tagger.predict(product)
    print(f"  Title:   {product['title'][:60]}")
    print(f"  Tags:    {result['tags']}")
    if result['hallucinations']:
        print(f"  [HALLUC] {result['hallucinations']}")
    if result['failed']:
        print(f"  [FAILED] Could not parse response")
    print()

## 3. Full Test Set Evaluation — Claude Haiku

In [ ]:
def evaluate_llm(provider: str, model: str, df: pd.DataFrame, label_names: list[str]) -> dict:
    """Run the full LLM evaluation loop and return metrics + telemetry."""
    tagger = LLMApiTagger(
        provider=provider,
        model=model,
        taxonomy_path='../data/taxonomy.json',
        cache_dir='../data/raw/llm_cache',
        request_delay=0.3,
    )

    all_preds, all_true = [], []
    latencies = []
    n = len(df)

    for i, row in df.iterrows():
        product = {'title': row['title'], 'description': row['description']}
        start   = time.perf_counter()
        result  = tagger.predict(product)
        latencies.append((time.perf_counter() - start) * 1000)

        all_preds.append(result['labels'])

        # Ground truth
        gt = row['labels']
        if isinstance(gt, str):
            import ast
            gt = ast.literal_eval(gt)
        all_true.append([int(v) for v in gt])

        if (i + 1) % 200 == 0:
            print(f'  {i+1}/{n} complete...')

    y_pred = np.array(all_preds)
    y_true = np.array(all_true)

    # LLM output is already binary — use 0.5 threshold (pass probs = binary pred)
    metrics  = compute_metrics(y_true, y_pred.astype(float), thresholds=[0.5] * len(label_names))
    failures = tagger.failure_report()
    cost     = tagger.cost_estimate(n_products=100_000)

    return {
        'metrics':   metrics,
        'failures':  failures,
        'cost':      cost,
        'latency':   {
            'mean_ms':   round(float(np.mean(latencies)), 2),
            'median_ms': round(float(np.median(latencies)), 2),
            'p95_ms':    round(float(np.percentile(latencies, 95)), 2),
            'p99_ms':    round(float(np.percentile(latencies, 99)), 2),
        },
    }


print('Evaluating Claude Haiku on test set...')
haiku_results = evaluate_llm(
    provider='anthropic',
    model='claude-haiku-4-5-20251001',
    df=test_df,
    label_names=LABEL_NAMES,
)

print('\nClaude Haiku — test set metrics:')
for k, v in haiku_results['metrics'].items():
    if k != 'per_label_f1':
        print(f'  {k:<20} {v}')

## 4. Full Test Set Evaluation — GPT-4o-mini

In [ ]:
print('Evaluating GPT-4o-mini on test set...')
gpt_results = evaluate_llm(
    provider='openai',
    model='gpt-4o-mini',
    df=test_df,
    label_names=LABEL_NAMES,
)

print('\nGPT-4o-mini — test set metrics:')
for k, v in gpt_results['metrics'].items():
    if k != 'per_label_f1':
        print(f'  {k:<20} {v}')

## 5. Three-Way Comparison Table

Assumes `bert_metrics` and `lora_metrics` are in scope from Phases 2 & 3.
If not, reload those checkpoints and re-evaluate before running this cell.

In [ ]:
try:
    all_results = {
        'BERT':           bert_metrics,
        'LoRA-Mistral':   lora_metrics,
        'Claude Haiku':   haiku_results['metrics'],
        'GPT-4o-mini':    gpt_results['metrics'],
    }

    comparison = compare_models(all_results, LABEL_NAMES)
    summary_df = pd.DataFrame(comparison['summary']).T

    print('All-model comparison (test set):')
    print(summary_df.to_string())

    summary_df.to_csv('../data/processed/all_model_comparison.csv')
    print('\nSaved to data/processed/all_model_comparison.csv')
except NameError:
    print('bert_metrics / lora_metrics not in scope — run Phases 2 & 3 first or reload checkpoints.')

## 6. Failure Mode Analysis

In [ ]:
for name, res in [('Claude Haiku', haiku_results), ('GPT-4o-mini', gpt_results)]:
    f = res['failures']
    n = len(test_df)
    print(f'{name}:')
    print(f'  Parse failures:    {f["parse_failure_count"]} / {n} ({100*f["parse_failure_count"]/n:.1f}%)')
    print(f'  Hallucinations:    {f["hallucination_count"]} / {n} ({100*f["hallucination_count"]/n:.1f}%)')
    print()

# Show example failures
print('Sample failure events (Claude Haiku):')
for ex in haiku_results['failures']['examples'][:5]:
    print(f"  Product:       {ex['product'][:50]}")
    if ex.get('failed'):
        print(f"  Type:          Parse failure")
        print(f"  Raw response:  {ex['raw'][:80]}")
    elif ex.get('hallucinations'):
        print(f"  Type:          Hallucination")
        print(f"  Unknown tags:  {ex['hallucinations']}")
    print()

## 7. Latency Variance

In [ ]:
print('Latency (end-to-end including network round-trip):')
for name, res in [('Claude Haiku', haiku_results), ('GPT-4o-mini', gpt_results)]:
    l = res['latency']
    print(f'  {name:<15} mean={l["mean_ms"]}ms  median={l["median_ms"]}ms  p95={l["p95_ms"]}ms  p99={l["p99_ms"]}ms')

# Load BERT and Mistral latencies for the full picture
try:
    with open('../data/processed/bert-base-uncased_latency.json') as f:
        bert_lat = json.load(f)
    with open('../data/processed/lora_mistral_latency.json') as f:
        lora_lat = json.load(f)

    print(f"  {'BERT':<15} mean={bert_lat['mean_ms']}ms  median={bert_lat['median_ms']}ms  p95={bert_lat['p95_ms']}ms  p99={bert_lat['p99_ms']}ms  (local GPU)")
    print(f"  {'LoRA-Mistral':<15} mean={lora_lat['mean_ms']}ms  median={lora_lat['median_ms']}ms  p95={lora_lat['p95_ms']}ms  p99={lora_lat['p99_ms']}ms  (local GPU)")
except FileNotFoundError:
    print('  (BERT / Mistral latency files not found — run Phases 2 & 3 first)')

## 8. Cost Analysis

In [ ]:
scale_points = [1_000, 10_000, 100_000, 1_000_000]

rows = []
for name, res in [('Claude Haiku', haiku_results), ('GPT-4o-mini', gpt_results)]:
    cost_fn = LLMApiTagger(
        provider='anthropic' if 'haiku' in name.lower() else 'openai',
        model='claude-haiku-4-5-20251001' if 'haiku' in name.lower() else 'gpt-4o-mini',
        taxonomy_path='../data/taxonomy.json',
        cache_dir='../data/raw/llm_cache',
    )
    # Seed token counts from actual evaluation
    cost_fn.total_input_tokens  = res['cost']['avg_input_tokens']
    cost_fn.total_output_tokens = res['cost']['avg_output_tokens']

    for n in scale_points:
        est = cost_fn.cost_estimate(n)
        rows.append({'Model': name, 'Products': f'{n:,}', 'Est. Cost (USD)': f"${est['estimated_cost_usd']:.2f}"})

cost_df = pd.DataFrame(rows).pivot(index='Products', columns='Model', values='Est. Cost (USD)')
print('Estimated API cost at scale:')
print(cost_df.to_string())

cost_df.to_csv('../data/processed/llm_cost_analysis.csv')

## 9. Per-Label F1 — All Models Heatmap

In [ ]:
try:
    heatmap_data = pd.DataFrame({
        'BERT':         bert_metrics['per_label_f1'],
        'LoRA-Mistral': lora_metrics['per_label_f1'],
        'Claude Haiku': haiku_results['metrics']['per_label_f1'],
        'GPT-4o-mini':  gpt_results['metrics']['per_label_f1'],
    }, index=LABEL_NAMES)

    heatmap_data['avg'] = heatmap_data.mean(axis=1)
    heatmap_data = heatmap_data.sort_values('avg', ascending=False).drop(columns='avg')

    fig, ax = plt.subplots(figsize=(10, 11))
    sns.heatmap(
        heatmap_data,
        annot=True, fmt='.2f',
        cmap='RdYlGn', vmin=0, vmax=1,
        linewidths=0.5, ax=ax,
    )
    ax.set_title('Per-label F1: All Models (test set)')
    plt.tight_layout()
    plt.savefig('../data/processed/per_label_f1_heatmap_all_models.png', dpi=150)
    plt.show()
except NameError:
    print('bert_metrics / lora_metrics not in scope — skipping full heatmap.')

In [ ]:
# Save all LLM results for Phase 5 reference
with open('../data/processed/llm_baseline_results.json', 'w') as f:
    json.dump({
        'claude_haiku': {
            'metrics': {k: v for k, v in haiku_results['metrics'].items() if k != 'per_label_f1'},
            'latency': haiku_results['latency'],
            'failures': haiku_results['failures'],
        },
        'gpt_4o_mini': {
            'metrics': {k: v for k, v in gpt_results['metrics'].items() if k != 'per_label_f1'},
            'latency': gpt_results['latency'],
            'failures': gpt_results['failures'],
        },
    }, f, indent=2)

print('Phase 4 complete.')
print('Results cached and saved to data/processed/llm_baseline_results.json')